In [1]:
import sys

te_path = "/home/kainingz/GitHub/TransformerEngine"

# Add it to the system path if it's not already there
if te_path not in sys.path:
    sys.path.append(te_path)

In [3]:
import os
import torch
import triton
import triton.language as tl
import triton.testing

# ---------------------------------------------------------------------------
# Import Triton ops (always available)
# ---------------------------------------------------------------------------
from transformer_engine.pytorch.triton.mhc import (
    mHCSinkhornOp,
    mHCAggregateOp,
    mHCExpandCombineOp,
    mHCProjectionOp,
)

# ---------------------------------------------------------------------------
# Import cuTile ops (only when cuda.tile is installed)
# ---------------------------------------------------------------------------
try:
    from cutile_kernels import (
        fused_sinkhorn,
        fused_h_aggregate,
        fused_h_post_bda,
        fused_proj_rms,
        is_cutile_available,
    )
    CUTILE_AVAILABLE = is_cutile_available()
except (ImportError, RuntimeError):
    CUTILE_AVAILABLE = False


OSError: /usr/local/lib/python3.12/dist-packages/transformer_engine/libtransformer_engine.so: undefined symbol: cublasLtGroupedMatrixLayoutInit_internal, version libcublasLt.so.13

In [ ]:
# ---------------------------------------------------------------------------
# Global config
# ---------------------------------------------------------------------------
DEVICE = "cuda"
DTYPE  = torch.bfloat16
N      = 4              # hyper-connection width; Triton kernels hard-require n=4
N_SINKHORN_ITERS = 20

# B=1 always (only B*T matters; vary T to sweep sequence length)
B_FIXED = 1
C_FIXED = 8192   # per-stream hidden dim (total hidden = n*C = 16384)

# Sweep range for T across all benchmarks
T_VALS = [512, 1024, 2048, 4096, 8192, 16384]

# Projection output features: 2n + n^2  with n=4  => 24
N_PROJ = 2 * N + N * N

# ---------------------------------------------------------------------------
# Helper: build line_vals / line_names / styles based on availability
# ---------------------------------------------------------------------------
def _lines():
    vals  = ["triton"] + (["cutile"] if CUTILE_AVAILABLE else [])
    names = ["Triton"] + (["cuTile"] if CUTILE_AVAILABLE else [])
    stys  = [("blue", "-")] + ([("red", "-")] if CUTILE_AVAILABLE else [])
    return vals, names, stys

_lv, _ln, _ls = _lines()

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["T"],
        x_vals=T_VALS,
        line_arg="provider",
        line_vals=_lv,
        line_names=_ln,
        styles=_ls,
        ylabel="Time (ms)",
        xlabel="T (sequence length)",
        plot_name=f"Sinkhorn_Forward_[B={B_FIXED}_n={N}_iters={N_SINKHORN_ITERS}]",
        args={},
    )
)
def bench_sinkhorn_fwd(T, provider):
    H_res = torch.randn(B_FIXED, T, N, N, dtype=DTYPE, device=DEVICE)

    if provider == "triton":
        fn = lambda: mHCSinkhornOp.apply(H_res, N, True, N_SINKHORN_ITERS)
    else:
        fn = lambda: fused_sinkhorn(H_res, num_iterations=N_SINKHORN_ITERS, eps=1e-6)

    return triton.testing.do_bench(fn)

In [ ]:
bench_sinkhorn_fwd.run(print_data=True, show_plots=True)

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["T"],
        x_vals=T_VALS,
        line_arg="provider",
        line_vals=_lv,
        line_names=_ln,
        styles=_ls,
        ylabel="Time (ms)",
        xlabel="T (sequence length)",
        plot_name=f"Sinkhorn_Backward_[B={B_FIXED}_n={N}_iters={N_SINKHORN_ITERS}]",
        args={},
    )
)
def bench_sinkhorn_bwd(T, provider):
    H_res = torch.randn(B_FIXED, T, N, N, dtype=DTYPE, device=DEVICE, requires_grad=True)

    if provider == "triton":
        out = mHCSinkhornOp.apply(H_res, N, True, N_SINKHORN_ITERS)
    else:
        out = fused_sinkhorn(H_res, num_iterations=N_SINKHORN_ITERS, eps=1e-6)

    loss = out.sum()
    fn   = lambda: loss.backward(retain_graph=True)
    return triton.testing.do_bench(fn)


In [ ]:
bench_sinkhorn_bwd.run(print_data=True, show_plots=True)

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["T"],
        x_vals=T_VALS,
        line_arg="provider",
        line_vals=_lv,
        line_names=_ln,
        styles=_ls,
        ylabel="Time (ms)",
        xlabel="T (sequence length)",
        plot_name=f"HAggregate_Forward_[B={B_FIXED}_C={C_FIXED}_n={N}]",
        args={},
    )
)
def bench_pre_fwd(T, provider):
    x_BTnC     = torch.randn(B_FIXED, T, N, C_FIXED, dtype=DTYPE, device=DEVICE)
    x_BTCn     = torch.randn(B_FIXED, T, C_FIXED, N, dtype=DTYPE, device=DEVICE)
    h_pre = torch.randn(B_FIXED, T, N,           dtype=DTYPE, device=DEVICE)

    if provider == "triton":
        fn = lambda: mHCAggregateOp.apply(x_BTCn, h_pre, N)
    elif provider == "cutile":
        fn = lambda: fused_h_aggregate(x_BTnC, h_pre)


    return triton.testing.do_bench(fn)

In [ ]:
bench_pre_fwd.run(print_data=True, show_plots=True)

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["T"],
        x_vals=T_VALS,
        line_arg="provider",
        line_vals=_lv,
        line_names=_ln,
        styles=_ls,
        ylabel="Time (ms)",
        xlabel="T (sequence length)",
        plot_name=f"HAggregate_Backward_[B={B_FIXED}_C={C_FIXED}_n={N}]",
        args={},
    )
)
def bench_pre_bwd(T, provider):
    x_BTnC     = torch.randn(B_FIXED, T, N, C_FIXED, dtype=DTYPE, device=DEVICE)
    x_BTCn     = torch.randn(B_FIXED, T, C_FIXED, N, dtype=DTYPE, device=DEVICE)
    h_pre = torch.randn(B_FIXED, T, N,           dtype=DTYPE, device=DEVICE, requires_grad=True)

    if provider == "triton":
        out = mHCAggregateOp.apply(x_BTCn, h_pre, N)
    else:
        out = fused_h_aggregate(x_BTnC, h_pre)

    loss = out.sum()
    fn   = lambda: loss.backward(retain_graph=True)
    return triton.testing.do_bench(fn)


In [ ]:
bench_pre_bwd.run(print_data=True, show_plots=True)

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["T"],
        x_vals=T_VALS,
        line_arg="provider",
        line_vals=_lv,
        line_names=_ln,
        styles=_ls,
        ylabel="Time (ms)",
        xlabel="T (sequence length)",
        plot_name=f"HPostBDA_PostResOp_Forward_[B={B_FIXED}_C={C_FIXED}_n={N}_bias=None]",
        args={},
    )
)
def bench_post_fwd_no_bias(T, provider):
    f      = torch.randn(B_FIXED, T,    C_FIXED, dtype=DTYPE, device=DEVICE)
    bias   = None
    H_post = torch.randn(B_FIXED, T, N,           dtype=DTYPE, device=DEVICE)
    x_BTnC = torch.randn(B_FIXED, T, N, C_FIXED,  dtype=DTYPE, device=DEVICE)
    H_res  = torch.randn(B_FIXED, T, N, N,         dtype=DTYPE, device=DEVICE)
    x_BTCn = torch.randn(B_FIXED, T, C_FIXED, N,  dtype=DTYPE, device=DEVICE)

    if provider == "triton":
        fn = lambda: mHCPostResOp.apply(f, bias, H_post, x_BTCn, H_res, N)
    else:
        # cuTile arg order: h_res, original_residual, h_post, x(=f), bias
        fn = lambda: fused_h_post_bda(H_res, x_BTnC, H_post, f, bias)

    return triton.testing.do_bench(fn)

In [ ]:
bench_post_fwd_no_bias.run(print_data=True, show_plots=True)

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["T"],
        x_vals=T_VALS,
        line_arg="provider",
        line_vals=_lv,
        line_names=_ln,
        styles=_ls,
        ylabel="Time (ms)",
        xlabel="T (sequence length)",
        plot_name=f"HPostBDA_PostResOp_Backward_[B={B_FIXED}_C={C_FIXED}_n={N}_bias=None]",
        args={},
    )
)
def bench_post_bwd_no_bias(T, provider):
    f      = torch.randn(B_FIXED, T,    C_FIXED, dtype=DTYPE, device=DEVICE, requires_grad=True)
    bias   = None
    H_post = torch.randn(B_FIXED, T, N,           dtype=DTYPE, device=DEVICE, requires_grad=True)
    x_BTnC      = torch.randn(B_FIXED, T, N, C_FIXED,  dtype=DTYPE, device=DEVICE, requires_grad=True)
    x_BTCn      = torch.randn(B_FIXED, T, C_FIXED, N,  dtype=DTYPE, device=DEVICE, requires_grad=True)
    H_res  = torch.randn(B_FIXED, T, N, N,         dtype=DTYPE, device=DEVICE, requires_grad=True)

    if provider == "triton":
        out = mHCExpandCombineOp.apply(f, bias, H_post, x_BTCn, H_res, N)
    else:
        out = fused_h_post_bda(H_res, x_BTnC, H_post, f, bias)

    loss = out.sum()
    fn   = lambda: loss.backward(retain_graph=True)
    return triton.testing.do_bench(fn)

In [ ]:
bench_post_bwd_no_bias.run(print_data=True, show_plots=True)

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["T"],
        x_vals=T_VALS,
        line_arg="provider",
        line_vals=_lv,
        line_names=_ln,
        styles=_ls,
        ylabel="Time (ms)",
        xlabel="T (sequence length)",
        plot_name=f"HPostBDA_PostResOp_Forward_[B={B_FIXED}_C={C_FIXED}_n={N}_bias=None]",
        args={},
    )
)
def bench_post_fwd(T, provider):
    f      = torch.randn(B_FIXED, T,    C_FIXED, dtype=DTYPE, device=DEVICE)
    bias   = torch.randn(C_FIXED, dtype=DTYPE, device=DEVICE)
    H_post = torch.randn(B_FIXED, T, N,           dtype=DTYPE, device=DEVICE)
    x_BTnC = torch.randn(B_FIXED, T, N, C_FIXED,  dtype=DTYPE, device=DEVICE)
    H_res  = torch.randn(B_FIXED, T, N, N,         dtype=DTYPE, device=DEVICE)
    x_BTCn = torch.randn(B_FIXED, T, C_FIXED, N,  dtype=DTYPE, device=DEVICE)

    if provider == "triton":
        fn = lambda: mHCPostResOp.apply(f, bias, H_post, x_BTCn, H_res, N)
    else:
        # cuTile arg order: h_res, original_residual, h_post, x(=f), bias
        fn = lambda: fused_h_post_bda(H_res, x_BTnC, H_post, f, bias)

    return triton.testing.do_bench(fn)

In [ ]:
bench_post_fwd.run(print_data=True, show_plots=True)

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["T"],
        x_vals=T_VALS,
        line_arg="provider",
        line_vals=_lv,
        line_names=_ln,
        styles=_ls,
        ylabel="Time (ms)",
        xlabel="T (sequence length)",
        plot_name=f"HPostBDA_PostResOp_Backward_[B={B_FIXED}_C={C_FIXED}_n={N}_bias=None]",
        args={},
    )
)
def bench_post_bwd(T, provider):
    f      = torch.randn(B_FIXED, T,    C_FIXED, dtype=DTYPE, device=DEVICE, requires_grad=True)
    bias   = torch.randn(C_FIXED, dtype=DTYPE, device=DEVICE)
    H_post = torch.randn(B_FIXED, T, N,           dtype=DTYPE, device=DEVICE, requires_grad=True)
    x_BTnC      = torch.randn(B_FIXED, T, N, C_FIXED,  dtype=DTYPE, device=DEVICE, requires_grad=True)
    x_BTCn      = torch.randn(B_FIXED, T, C_FIXED, N,  dtype=DTYPE, device=DEVICE, requires_grad=True)
    H_res  = torch.randn(B_FIXED, T, N, N,         dtype=DTYPE, device=DEVICE, requires_grad=True)

    if provider == "triton":
        out = mHCExpandCombineOp.apply(f, bias, H_post, x_BTCn, H_res, N)
    else:
        out = fused_h_post_bda(H_res, x_BTnC, H_post, f, bias)

    loss = out.sum()
    fn   = lambda: loss.backward(retain_graph=True)
    return triton.testing.do_bench(fn)

In [ ]:
bench_post_bwd.run(print_data=True, show_plots=True)

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["T"],
        x_vals=T_VALS,
        line_arg="provider",
        line_vals=_lv,
        line_names=_ln,
        styles=_ls,
        ylabel="Time (ms)",
        xlabel="T (sequence length)",
        plot_name=(
            f"ProjRms_ProjectionOp_Forward_[B={B_FIXED}_C={C_FIXED}_n={N}_N_proj={N_PROJ}]"
        ),
        args={},
    )
)
def bench_proj_fwd(T, provider):
    M   = B_FIXED * T
    K   = N * C_FIXED  # total hidden dimension
    x   = torch.randn(M, K,      dtype=DTYPE, device=DEVICE)
    phi = torch.randn(N_PROJ, K, dtype=DTYPE, device=DEVICE)

    if provider == "triton":
        fn = lambda: mHCProjectionOp.apply(x, phi)
    else:
        fn = lambda: fused_proj_rms(x, phi)

    return triton.testing.do_bench(fn)

In [ ]:
bench_proj_fwd.run(print_data=True, show_plots=True)

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["T"],
        x_vals=T_VALS,
        line_arg="provider",
        line_vals=_lv,
        line_names=_ln,
        styles=_ls,
        ylabel="Time (ms)",
        xlabel="T (sequence length)",
        plot_name=(
            f"ProjRms_ProjectionOp_Backward_[B={B_FIXED}_C={C_FIXED}_n={N}_N_proj={N_PROJ}]"
        ),
        args={},
    )
)
def bench_proj_bwd(T, provider):
    M   = B_FIXED * T
    K   = N * C_FIXED
    x   = torch.randn(M, K,      dtype=DTYPE, device=DEVICE, requires_grad=True)
    phi = torch.randn(N_PROJ, K, dtype=DTYPE, device=DEVICE, requires_grad=True)

    if provider == "triton":
        H, r = mHCProjectionOp.apply(x, phi)
        loss = H.sum() + r.sum()
    else:
        proj, r = fused_proj_rms(x, phi)
        loss = proj.sum() + r.sum()

    fn = lambda: loss.backward(retain_graph=True)
    return triton.testing.do_bench(fn)

In [ ]:
bench_proj_bwd.run(print_data=True, show_plots=True)